# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### RunnableLambda
일반 Python 함수를 lcel 체인에서 사용할 수 있는 Runnable 형태로 wrapping 처리해주는 클래스

In [ ]:
# 입력을 받아 내장된 함수를 실행하는 Runnable
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕, 만나서 반가워')

11

In [4]:
# batch() : 여러 건의 입력을 일괄처리해줌
runnable.batch(['안녕, 만나서 반가워', '너도? 나도야', 'ㅎㅎ', '😻😻'])

[11, 7, 2, 2]

In [5]:
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

celsius_temp = [0, 25, 100, -10, 37]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temp)

[32.0, 77.0, 212.0, 14.0, 98.6]

In [ ]:
import time  # 출력 딜레이용

def generator(x):
    for y in x:  # 입력을 문자 단위로 순회
        yield y  # 한 글자씩 반환 (스트리밍 방식)

runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️'):
    print(chunk, end='', flush=True)  # chunk를 줄바꿈없이 즉시 출력
    time.sleep(0.1)  # 글자 출력마다 딜레이 0.1초

안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️안녕하세요-!🥹♥️♥️

In [9]:
# 사용 예시
def gen(x):
    for y in x:
        yield y

gen10 = gen(range(10))

for n in gen10:
    print(n)

0
1
2
3
4
5
6
7
8
9


In [ ]:
next(gen10)  # 제너레이터 다음 값 1개 반환 (다 꺼내고나면 StopIteration 발생)

StopIteration: 

In [11]:
gen10 = gen(range(10))
next(gen10)

0

### RunnabelSequence
Runnable 객체를 순차연결해주는 Runnable 객체

In [ ]:
from langchain_core.runnables import RunnableSequence  # Runnable들을 순서대로 연결하는 시퀀스

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableSequence(runnable1, runnable2)  # runnabel1 -> runnabel2
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [13]:
chain = runnable1 | runnable2
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [14]:
chain = runnable2 | runnable1
chain.invoke(3)

{'foo': [3, 3, 3]}

### RunnableParallel
여러 Runnabel 객체를 인자로 받아, 병렬처리 후 각각의 응답을 하나의 dict로 반환

In [15]:
from langchain_core.runnables import RunnableParallel  # 여러 Runnable들을 같은 입력으로 병렬로 실행

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableParallel(r1=runnable1, r2=runnable2)  # r1과 r2를 병렬 실행해 dict로 반환
chain.invoke(3)

{'r1': {'foo': 3}, 'r2': [3, 3, 3]}

- 사용자가 주는 주제를 이용해서 삼행시, 농담, 시를 각각 생성해서 하나의 응답으로 반환

In [ ]:
from langchain_core.prompts import PromptTemplate  # 프롬프트 체인 구성 시 사용
from langchain.chat_models import init_chat_model  # 모델 체인 구성 래퍼 클래스
from langchain_core.output_parsers import StrOutputParser  # 답변 문자열 반환
from langchain_core.runnables import RunnableParallel  # 여러 Runnable들을 같은 입력으로 병렬로 실행

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser

joke_prompt = PromptTemplate.from_template(
    '당신은 엄청난 한국 개그맨입니다. 다음 주제로 누구에게나 웃긴 농담을 지어주세요. 주제 : {topic}'
)
joke_chain = joke_prompt | llm | output_parser

poem_prompt = PromptTemplate.from_template(
    '당신은 유명한 작가입니다. 다음 주제로 감성적인 시를 지어주세요. 주제 : {topic}'
)
poem_chain = poem_prompt | llm | output_parser

# 동일 입력(topic)을 받아 3개 체인을 병렬 실행  {acrostic_poem: 실행결과, ...}
chain = RunnableParallel(
    acrostic_poem = n_poem_chain,
    joke = joke_chain,
    poem = poem_chain
)

def combine_result(input_dict: dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem = input_dict['poem']
    return f"""
    n행시 : 
    {acrostic_poem} 

    농담 : 
    {joke}  

    현대시 : 
    {poem}  
    """

chain = chain | RunnableLambda(combine_result)
print(chain.invoke({'topic': '여행'}))


    n행시 : 
    **여**: 여기서 망설이면 평생 사진으로만 보게 되고  
**행**: 행복을 찾으려면 일단 가방부터 싸야지! 

    농담 : 
    물론이죠! 여행 주제로 몇 개 가겠습니다. ✈️

1. **여행 가서 제일 많이 듣는 말은?**  
   “여권 보여주세요.”  
   그런데 집에 돌아오면 가족이 말합니다.  
   “여권은 됐고, 카드값 보여줘.”

2. **여행 중 길을 잃은 사람이 가장 먼저 찾는 곳은?**  
   길이 아니라… **검색 기록 삭제하는 곳.**  
   “나 길 안 잃었어. 그냥 새로운 길을 개척한 거야.”

3. **비행기에서 가장 인기 많은 사람은 누구일까요?**  
   승무원? 기장?  
   아니요. **창가 자리 양보해주는 사람.**  
   그 순간부터 전 국민의 친척이 됩니다.  
   “아이고, 선생님! 복 받으세요!”

4. **여행 가서 사진을 500장 찍었는데 건진 사진이 한 장도 없는 이유는?**  
   풍경은 멋졌는데…  
   내가 계속 **“잠깐만, 다시 찍어!”**를 외쳤기 때문입니다.

5. **해외여행에서 가장 당황스러운 순간은?**  
   현지인이 영어로 길을 물어봤을 때.  
   여행자는 속으로 생각합니다.  
   “저도 관광객인데요… 왜 저한테 물어보세요? 얼굴이 구글맵인가요?”

6. **여행 계획을 완벽하게 세운 사람의 특징은?**  
   첫날 아침 9시 일정까지만 완벽합니다.  
   오후 2시부터는 계획이 아니라 **생존 다큐멘터리**가 됩니다.

7. **여행 가서 꼭 사 오는 기념품은?**  
   냉장고 자석, 엽서, 열쇠고리…  
   그리고 아무도 부탁하지 않았는데 사 온 **이상한 모자**.  
   “이거 현지에서 유행이야!”  
   — 집에 오면 아무도 안 씁니다.

8. **여행의 진짜 묘미는 뭘까요?**  
   새로운 문화를 경험하는 것?  
   아름다운 풍경을 보는 것?  
   아닙니다.  
   **집에 돌아와서 내 침대

### RunnablePassthrough
- 사용자의 입력값을 그대로 전달해주는 Runnable

In [18]:
acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser

print(n_poem_chain.invoke({'topic': '아르바이트'}))

아: 아침부터 부지런히 출근해  
르: 르네상스급 미소로 손님을 맞이하고  
바: 바쁜 순간에도 실수 없이 척척!  
이: 이렇게 하루를 보람차게 채우면  
트: 퇴근 후 통장 잔고가 나를 반긴다!


In [19]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template(
    '당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
chain = {'topic': RunnablePassthrough()} | prompt | llm | output_parser

print(chain.invoke('놀이공원'))

**놀**라운 비명이 하늘을 가르고  
**이** 순간만큼은 어른도 아이가 되어  
**공**중을 나는 롤러코스터에 마음을 맡기면  
**원** 없이 행복한 하루가 시작된다!


In [ ]:
prompt = PromptTemplate.from_template("""
    당신은 n행시의 엄청난 고수입니다. 다음 주제로 n행시를 지어주세요.

    주제 : {topic}
    
    출력 형식 :  
    === <주제> <{n}행시> ===
    <{n}행시 작성>
""")
chain = ({'topic': RunnablePassthrough()} 
    | RunnablePassthrough.assign(  # 기존 topic만 있단 dict -> 새 key를 추가
        n= lambda x: len(x['topic']),  # n = topic 길이
        k= lambda x: 100
    )
    | prompt 
    | llm 
    | output_parser
)

print(chain.invoke('아이스크림'))

=== 아이스크림 5행시 ===  
아이스크림 하나에  
이렇게 행복해질 줄이야  
스르르 녹는 달콤함처럼  
크게 웃음이 번지고  
림처럼 둥근 하루가 완성된다
